# Dublin Bikes Bronze Processing

Read raw Dublin Bikes data from the ADLS Landing zone, apply minimal transformations, and load it into the Bronze Delta table for the Urban Mobility platform.

In [0]:
from pyspark.sql.functions import current_timestamp, to_timestamp, when

In [0]:
storage_account = "dubmobdev002"
container = "landing"
dataset = "dublinbikes"
year = "2026"
month = "05"

In [0]:
landing_path = (
    f"abfss://{container}@{storage_account}.dfs.core.windows.net/"
    f"{dataset}/year={year}/month={month}"
)

df = (
    spark.read
         .option("header", "true")
         .csv(landing_path)
)

In [0]:
df.printSchema()

In [0]:
display(df.limit(5))

In [0]:
print(f"Rows: {df.count()}")


In [0]:
from pyspark.sql.functions import col, current_timestamp, to_timestamp

bronze_df = (
    df
    .drop("short_name", "region_id")
    .withColumn("last_reported", to_timestamp(col("last_reported")))
    .withColumn("lat", col("lat").cast("double"))
    .withColumn("lon", col("lon").cast("double"))
    .withColumn("capacity", col("capacity").cast("int"))
    .withColumn("num_bikes_available", col("num_bikes_available").cast("int"))
    .withColumn("num_docks_available", col("num_docks_available").cast("int"))
    .withColumn("is_installed", col("is_installed").cast("boolean"))
    .withColumn("is_renting", col("is_renting").cast("boolean"))
    .withColumn("is_returning", col("is_returning").cast("boolean"))
    .withColumn("processed_timestamp", current_timestamp())
)

In [0]:
bronze_df.printSchema()
display(bronze_df.limit(5))

In [0]:
(
    bronze_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("urban_mobility.bronze.stations_raw")
)

In [0]:
%sql
SELECT * FROM urban_mobility.bronze.stations_raw
LIMIT 5;